# Decisiveness Benchmark Public API Demo

This notebook demonstrates the release-facing decisiveness benchmark API using local frozen artifacts.


In [ ]:
import pandas as pd

from polymarket_research.benchmarks import evaluate_decisiveness, load_decisiveness
from polymarket_research.benchmarks.baselines import fit_decisiveness_majority_baseline
from polymarket_research.benchmarks.audit.reporting import benchmark_manifest_summary
from polymarket_research.benchmarks.visualization.plotting import (
    plot_confusion_matrix,
    plot_label_distribution,
    plot_market_history,
    plot_numeric_distribution,
)

from polymarket_research.utils.filesystem import setup_root

pd.set_option("display.max_columns", 100)

REPO_ROOT = setup_root()
ARTIFACT_ROOT = REPO_ROOT / "benchmark_releases"
VERSION = "v1"


In [ ]:
polymarket_bundle_dir = ARTIFACT_ROOT / "polymarket" / "decisiveness" / VERSION
polymarket_decisiveness = load_decisiveness(polymarket_bundle_dir)

kalshi_bundle_dir = ARTIFACT_ROOT / "kalshi" / "decisiveness" / VERSION
kalshi_decisiveness = load_decisiveness(kalshi_bundle_dir)

decisiveness_benchmarks = {
    "polymarket": polymarket_decisiveness,
    "kalshi": kalshi_decisiveness,
}

benchmark_manifest_summary(decisiveness_benchmarks)


## Polymarket


In [ ]:
benchmark = decisiveness_benchmarks["polymarket"]

baseline = fit_decisiveness_majority_baseline(benchmark, split="train")
predictions = baseline.predict(benchmark, split="test")
report = evaluate_decisiveness(benchmark, predictions, split="test")

pd.Series(baseline.manifest(), dtype="object")


In [ ]:
display(report["overall"])
display(report["continuous_overall"])
display(predictions.head())


## Kalshi


In [ ]:
benchmark = decisiveness_benchmarks["kalshi"]

baseline = fit_decisiveness_majority_baseline(benchmark, split="train")
predictions = baseline.predict(benchmark, split="test")
report = evaluate_decisiveness(benchmark, predictions, split="test")

pd.Series(baseline.manifest(), dtype="object")


In [ ]:
display(report["overall"])
display(report["continuous_overall"])
display(predictions.head())


## Side-by-side Baseline Summary


In [ ]:
summary_rows = []

for source, benchmark in decisiveness_benchmarks.items():
    baseline = fit_decisiveness_majority_baseline(benchmark, split="train")
    predictions = baseline.predict(benchmark, split="test")
    report = evaluate_decisiveness(benchmark, predictions, split="test")
    row = report["overall"].iloc[0].to_dict()
    row["continuous_mae_hours"] = float(report["continuous_overall"].iloc[0]["mae"])
    row["source"] = source
    row["baseline"] = baseline.manifest()["name"]
    summary_rows.append(row)

pd.DataFrame(summary_rows).loc[:, ["source", "baseline", "split", "rows", "accuracy", "macro_f1", "ordinal_mae", "continuous_mae_hours"]]


## Visual Checks

These plots use only frozen artifacts, baseline predictions, and evaluator reports.


In [ ]:
benchmark = decisiveness_benchmarks["polymarket"]
baseline = fit_decisiveness_majority_baseline(benchmark, split="train")
predictions = baseline.predict(benchmark, split="test")

plot_label_distribution(benchmark, split="test", title="Polymarket decisiveness test labels")
plot_numeric_distribution(benchmark, column="hours_to_decisive", split="test", title="Polymarket hours to decisive")
plot_confusion_matrix(benchmark, predictions, split="test", normalize=True, title="Polymarket decisiveness confusion matrix")


## Optional Inspection


In [ ]:
benchmark = decisiveness_benchmarks["polymarket"]
example = benchmark.input_frame().iloc[0]
market_id = str(example["market_id"])
cutoff_timestamp_utc = pd.Timestamp(example["cutoff_timestamp_utc"])
target_row = benchmark.targets().loc[
    lambda df: (df["market_id"].astype(str) == market_id)
    & (pd.to_datetime(df["cutoff_timestamp_utc"], utc=True, errors="coerce").eq(cutoff_timestamp_utc))
].iloc[0]

plot_market_history(
    benchmark,
    market_id=market_id,
    cutoff=cutoff_timestamp_utc,
    title="Polymarket example decisiveness history",
)
example_display = pd.concat([
    benchmark.resolve_market_snapshot(market_id, cutoff_timestamp_utc),
    target_row.drop(labels=["market_id", "cutoff_timestamp_utc", "split"], errors="ignore"),
])
display(example_display.loc[["market_id", "cutoff_timestamp_utc", "label_name", "hours_to_decisive"]])
benchmark.history_until(market_id, cutoff_timestamp_utc).tail()
